# Clase 13 · Casos de uso reales y cierre del curso

**Curso:** Advanced Deep Learning (aprendizaje profundo avanzado)

Bienvenido a la última clase del curso. Hoy no aprendes una técnica nueva: repasas en código las piezas que ya dominas y las conectas con sistemas reales de inteligencia artificial (IA) que cambiaron su campo, tal como los presenta el capítulo 14 del manual, *Casos de uso reales*.

## Objetivos de la sesión de cierre

Al final de esta clase, eres capaz de:

- Repasar, de un vistazo, las 13 clases del curso y la técnica central de cada una
- Cargar un modelo de visión preentrenado y usarlo para clasificar una imagen real, como hacen los sistemas de visión por computador en producción
- Revisitar la red generativa antagónica (GAN) de la Clase 9 y ver de nuevo el generador y el discriminador compitiendo
- Revisitar la atención escalada por producto punto de la Clase 8 y observar su mapa de calor sobre un ejemplo nuevo
- Identificar próximos pasos concretos para seguir formándote en deep learning por tu cuenta

## Repaso relámpago del curso

Antes de entrar en código, repasa en un vistazo qué aprendiste en cada una de las 13 clases:

- **Clase 1 — Introducción a la IA avanzada y tu primera red neuronal.** Qué hace avanzado al deep learning y tu primer perceptrón multicapa con retropropagación (backpropagation)
- **Clase 2 — Visión por computador I: redes convolucionales.** Filtros, stride, padding y pooling en una red neuronal convolucional (CNN)
- **Clase 3 — Visión por computador II: arquitecturas avanzadas.** Bloques residuales, degradación en redes profundas y tu primer modelo preentrenado con transferencia de aprendizaje
- **Clase 4 — Optimización I.** Descenso de gradiente, tasa de aprendizaje y convergencia
- **Clase 5 — Optimización II.** Momento, métodos de segundo orden, Adam y regularización
- **Clase 6 — Secuencias y redes recurrentes.** Redes neuronales recurrentes (RNN), el problema del gradiente que desaparece o explota, y cómo la memoria a largo y corto plazo (LSTM) lo resuelve con puertas
- **Clase 7 — Deep learning para lenguaje natural.** Tokenización, embeddings y un clasificador de texto, la base del procesamiento del lenguaje natural (PLN)
- **Clase 8 — Atención y Transformers.** Consulta, clave y valor, y la atención escalada por producto punto implementada desde cero
- **Clase 9 — Redes generativas antagónicas.** El generador y el discriminador de una GAN compitiendo en un juego minimax
- **Clase 10 — Aprendizaje no supervisado.** Autoencoders y representaciones aprendidas sin etiquetas
- **Clase 11 — Modelos generativos modernos.** El autocodificador variacional (VAE), el truco de reparametrización y el espacio latente
- **Clase 12 — IA responsable.** Sesgo, equidad, interpretabilidad y gobernanza
- **Clase 13 — Casos de uso reales y cierre.** Hoy: conectas todo lo anterior con sistemas reales y decides tus próximos pasos

## Visión: un modelo preentrenado en acción

El capítulo 14 del manual conecta la visión por computador con casos como AlphaFold, que combina bloques residuales y atención para predecir la forma de una proteína, y con cualquier sistema industrial que clasifica imágenes en producción. Ya construiste una red neuronal convolucional desde cero en la Clase 2 y usaste un modelo preentrenado en la Clase 3. Repite ahora ese mismo patrón en segundos: cargas una ResNet-18 entrenada sobre ImageNet y la usas para clasificar una imagen nueva, sin entrenar nada.

In [ ]:
import urllib.request

import torch
from PIL import Image
from torchvision import models, transforms

# Carga la ResNet-18 con los pesos entrenados sobre ImageNet
modelo_vision = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
modelo_vision.eval()  # desactiva el dropout y fija las estadísticas de BatchNorm

# Descarga una imagen de ejemplo y la lista de las 1000 clases de ImageNet
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg",
    "imagen_ejemplo.jpg",
)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt",
    "clases_imagenet.txt",
)
with open("clases_imagenet.txt") as archivo:
    categorias = [linea.strip() for linea in archivo.readlines()]

imagen = Image.open("imagen_ejemplo.jpg").convert("RGB")

preprocesar = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])
entrada = preprocesar(imagen).unsqueeze(0)

with torch.no_grad():
    salida = modelo_vision(entrada)
probabilidades = torch.nn.functional.softmax(salida[0], dim=0)

top5_prob, top5_idx = torch.topk(probabilidades, 5)

print("Predicciones de la ResNet-18 preentrenada:\n")
for probabilidad, indice in zip(top5_prob, top5_idx):
    print(f"{categorias[indice]:25s} {probabilidad.item() * 100:5.1f} %")

imagen

## Generación: tu GAN de la Clase 9 revisitada

El capítulo 14 conecta la generación de datos sintéticos con StyleGAN, que genera rostros que no existen, y con el uso de GAN para ampliar conjuntos de datos médicos escasos sin comprometer la privacidad de ningún paciente. Repite ahora, en pocos pasos, el mismo experimento de la Clase 9: un generador y un discriminador que compiten hasta que las muestras generadas se acercan a la distribución real.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- Datos reales: mezcla de ocho gaussianas en círculo, igual que en la Clase 9 ---
def sample_real_data(n_samples, n_modes=8, radius=4.0, std=0.15):
    angles = 2 * np.pi * np.random.randint(0, n_modes, size=n_samples) / n_modes
    centers = np.stack([radius * np.cos(angles), radius * np.sin(angles)], axis=1)
    points = centers + std * np.random.randn(n_samples, 2)
    return torch.tensor(points, dtype=torch.float32)


LATENT_DIM = 2


# --- Generador: ruido latente -> punto en dos dimensiones ---
class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, z):
        return self.net(z)


# --- Discriminador: punto en dos dimensiones -> logit de "realidad" ---
class Discriminator(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)


generator = Generator().to(device)
discriminator = Discriminator().to(device)
criterion = nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))

BATCH_SIZE = 256
N_STEPS = 1500  # entrenamiento corto: hoy repasas el patrón, no repites el entrenamiento completo de la Clase 9

for step in range(1, N_STEPS + 1):
    # 1) Actualiza el discriminador
    real_data = sample_real_data(BATCH_SIZE).to(device)
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    fake_data = generator(z).detach()

    real_labels = torch.ones(BATCH_SIZE, 1, device=device)
    fake_labels = torch.zeros(BATCH_SIZE, 1, device=device)

    loss_d = criterion(discriminator(real_data), real_labels) + criterion(discriminator(fake_data), fake_labels)
    opt_d.zero_grad()
    loss_d.backward()
    opt_d.step()

    # 2) Actualiza el generador con la pérdida no saturante
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    fake_data = generator(z)
    loss_g = criterion(discriminator(fake_data), real_labels)
    opt_g.zero_grad()
    loss_g.backward()
    opt_g.step()

    if step % 500 == 0:
        print(f"Paso {step} de {N_STEPS} | pérdida D: {loss_d.item():.3f} | pérdida G: {loss_g.item():.3f}")

# Muestra final: datos reales frente a datos generados tras el entrenamiento corto
with torch.no_grad():
    real_vis = sample_real_data(500).numpy()
    fake_vis = generator(torch.randn(500, LATENT_DIM, device=device)).cpu().numpy()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(real_vis[:, 0], real_vis[:, 1], s=10, alpha=0.5, label="Datos reales")
ax.scatter(fake_vis[:, 0], fake_vis[:, 1], s=10, alpha=0.5, label="Datos generados")
ax.set_title(f"GAN revisitada - tras {N_STEPS} pasos de entrenamiento")
ax.legend(loc="upper right")
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_aspect("equal")
plt.show()

## Atención: mirar donde importa

El capítulo 14 conecta la atención con la traducción automática moderna y con AlphaFold, que la usa para relacionar aminoácidos lejanos en la secuencia pero cercanos en la estructura tridimensional final. Repite ahora el cálculo de la Clase 8 —consulta, clave y valor— sobre una frase nueva, y observa de nuevo el mapa de calor de los pesos de atención.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(1)

# Frase nueva, distinta a la de la Clase 8, ya tokenizada en palabras
tokens = ["la", "traduccion", "automatica", "conecta", "idiomas", "distintos"]
n = len(tokens)
d = 8      # dimensión del embedding, pequeña solo para que el ejemplo sea legible
d_k = 8    # dimensión de consultas y claves

X = torch.randn(n, d)
W_q = torch.randn(d, d_k) * 0.5
W_k = torch.randn(d, d_k) * 0.5
W_v = torch.randn(d, d_k) * 0.5

Q = X @ W_q   # consultas: qué busca cada palabra
K = X @ W_k   # claves: qué ofrece cada palabra al ser comparada
V = X @ W_v   # valores: qué contenido entrega cada palabra si es elegida


def softmax_manual(z, dim=-1):
    z = z - z.max(dim=dim, keepdim=True).values
    exp_z = torch.exp(z)
    return exp_z / exp_z.sum(dim=dim, keepdim=True)


def atencion(Q, K, V):
    d_k = Q.shape[-1]
    puntuaciones = Q @ K.transpose(-2, -1)          # producto punto: consulta contra cada clave
    puntuaciones = puntuaciones / (d_k ** 0.5)       # escalado por raíz de d_k
    pesos = softmax_manual(puntuaciones, dim=-1)     # cada fila suma uno: es una distribución
    salida = pesos @ V                               # combinación ponderada de los valores
    return salida, pesos


salida, pesos_atencion = atencion(Q, K, V)

plt.figure(figsize=(6, 5))
plt.imshow(pesos_atencion.detach().numpy(), cmap="viridis")
plt.xticks(range(n), tokens, rotation=45, ha="right")
plt.yticks(range(n), tokens)
plt.xlabel("Clave (palabra observada)")
plt.ylabel("Consulta (palabra que pregunta)")
plt.title("Atención escalada por producto punto, revisitada")
plt.colorbar(label="Peso de atención")
plt.tight_layout()
plt.show()

## Y ahora, ¿qué?

Terminar este curso no es el final del camino, es el punto de partida. Estas son formas concretas de seguir aprendiendo:

- **Lee papers originales.** Empieza por *Attention Is All You Need* (el Transformer), *Generative Adversarial Networks* (Goodfellow y colaboradores), *Deep Residual Learning for Image Recognition* (ResNet) y el informe técnico de AlphaFold publicado en Nature. No necesitas entender cada ecuación a la primera lectura: lee la introducción y las figuras, y vuelve a las matemáticas cuando las necesites
- **Únete a una comunidad.** Papers With Code enlaza cada paper con su implementación, Hugging Face reúne modelos, conjuntos de datos y una comunidad activa, y los foros y grupos locales de IA te conectan con personas que resuelven problemas parecidos a los tuyos
- **Construye un proyecto propio.** Elige un problema real de tu entorno —agroalimentario, forestal, salud, biotecnológico o cualquier otro que conozcas de cerca— y aplícale una de las técnicas de este curso. Nada enseña tanto como depurar tu propio código a las 11 de la noche
- **Vuelve al capítulo 13 cuando despliegues algo real.** Antes de poner en producción cualquier modelo, revisa el sesgo, la robustez frente a entradas inesperadas y el impacto sobre las personas afectadas

**Sigue leyendo papers, sigue experimentando con código y sigue preguntándote no solo cómo funciona un modelo, sino para qué y para quién lo estás construyendo.**

## Resumen de puntos clave del curso completo

- Dominaste los fundamentos: neuronas, capas, retropropagación y descenso de gradiente, con y sin momento
- Construiste redes neuronales convolucionales y aprendiste por qué las conexiones residuales permiten entrenar redes muy profundas
- Modelaste secuencias con redes neuronales recurrentes y memoria a largo y corto plazo, y superaste su cuello de botella con atención
- Implementaste desde cero la atención escalada por producto punto, la pieza central del Transformer
- Entrenaste una red generativa antagónica y un autocodificador variacional, dos formas distintas de generar datos nuevos
- Practicaste aprendizaje no supervisado con autoencoders y representaciones aprendidas sin etiquetas
- Evaluaste sesgo, equidad e interpretabilidad, y aprendiste a desplegar modelos de forma responsable
- Viste hoy cómo estas mismas piezas —convoluciones, atención, generación adversaria— se combinan en sistemas reales como AlphaGo, AlphaFold, el traductor neuronal de Google, WaveNet, StyleGAN y los grandes modelos de lenguaje modernos

**Abre ahora el notebook de práctica companion, `../notebooks_practica/clase13_practica_casos_de_uso_cierre.ipynb`, y cierra el curso poniendo en práctica todo lo que aprendiste.**